In [6]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

df = pd.read_csv("parkinsons.csv")

print(df.head())
print(df.info())
print(df.shape)
print(df.columns)

df = df.dropna()
df = df.drop(columns=['name'], errors = 'ignore')

y = df['status']
x = df.drop(columns=['status'])

for col in x.select_dtypes(include="object").columns:
    x[col] = LabelEncoder().fit_transform(x[col])

if y.dtype == "object":
    y = LabelEncoder().fit_transform(y)

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

feature_names = x.columns.tolist()
print("Shape:", x_scaled.shape, "Classes:", set(y))

             name  MDVP:Fo(Hz)  MDVP:Fhi(Hz)  MDVP:Flo(Hz)  MDVP:Jitter(%)  \
0  phon_R01_S01_1      119.992       157.302        74.997         0.00784   
1  phon_R01_S01_2      122.400       148.650       113.819         0.00968   
2  phon_R01_S01_3      116.682       131.111       111.555         0.01050   
3  phon_R01_S01_4      116.676       137.871       111.366         0.00997   
4  phon_R01_S01_5      116.014       141.781       110.655         0.01284   

   MDVP:Jitter(Abs)  MDVP:RAP  MDVP:PPQ  Jitter:DDP  MDVP:Shimmer  ...  \
0           0.00007   0.00370   0.00554     0.01109       0.04374  ...   
1           0.00008   0.00465   0.00696     0.01394       0.06134  ...   
2           0.00009   0.00544   0.00781     0.01633       0.05233  ...   
3           0.00009   0.00502   0.00698     0.01505       0.05492  ...   
4           0.00011   0.00655   0.00908     0.01966       0.06425  ...   

   Shimmer:DDA      NHR     HNR  status      RPDE       DFA   spread1  \
0      0.0654

In [3]:
import numpy as np 
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedKFold, cross_val_score

random_state = 42
models = {
    "Linear Classifier": Perceptron(max_iter=1000, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state = 42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Gaussian NB": GaussianNB(),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42),
}

rkf = RepeatedKFold(n_splits=10, n_repeats=100, random_state=42)

results_part2 = {}
for name, model in models.items():
    scores = cross_val_score(model, x_scaled, y,cv=rkf, scoring="accuracy", n_jobs=-1)
    results_part2[name] = (scores.mean(), scores.std())
    print(f"{name:20s} mean={scores.mean():.4f} std={scores.std():.4f}")


Linear Classifier    mean=0.8085 std=0.0926
Logistic Regression  mean=0.8535 std=0.0780
KNN                  mean=0.9110 std=0.0657
Gaussian NB          mean=0.6973 std=0.0995


/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/python/3.12.1/lib

Neural Network       mean=0.9254 std=0.0594


/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
import numpy as np

def best_subset_for_model(model, x_scaled, y, feature_names):
    sfs = SequentialFeatureSelector(
        estimator=model,
        n_features_to_select="auto",   
        direction="forward",           
        scoring="accuracy",
        cv= RepeatedKFold(n_splits=10, n_repeats=10, random_state=42),
        n_jobs=-1,
    )

    sfs.fit(x_scaled, y)
    mask = sfs.get_support()
    selected_features = np.array(feature_names)[mask]

    X_sel = x_scaled[:, mask]
    scores = cross_val_score(model, X_sel, y, cv= RepeatedKFold(n_splits=10, n_repeats=3, random_state=42), scoring="accuracy", n_jobs=-1)
    return list(selected_features), scores.mean(), scores.std()

results_part3 = {}
for name, model in models.items():
    subset, mean_acc, std_acc = best_subset_for_model(model, x_scaled, y, feature_names)
    results_part3[name] = (subset, mean_acc, std_acc)
    print(f"{name}\n  Features: {subset}\n  mean={mean_acc:.4f} std={std_acc:.4f}\n")


Linear Classifier
  Features: [np.str_('MDVP:Fo(Hz)'), np.str_('MDVP:RAP'), np.str_('MDVP:Shimmer'), np.str_('MDVP:Shimmer(dB)'), np.str_('Shimmer:APQ3'), np.str_('Shimmer:APQ5'), np.str_('MDVP:APQ'), np.str_('NHR'), np.str_('RPDE'), np.str_('spread1'), np.str_('D2')]
  mean=0.8096 std=0.1029

Logistic Regression
  Features: [np.str_('MDVP:Fo(Hz)'), np.str_('MDVP:Fhi(Hz)'), np.str_('MDVP:Flo(Hz)'), np.str_('MDVP:Jitter(%)'), np.str_('MDVP:Jitter(Abs)'), np.str_('MDVP:RAP'), np.str_('MDVP:PPQ'), np.str_('RPDE'), np.str_('spread1'), np.str_('spread2'), np.str_('PPE')]
  mean=0.8713 std=0.0819

KNN
  Features: [np.str_('MDVP:Fo(Hz)'), np.str_('MDVP:RAP'), np.str_('MDVP:PPQ'), np.str_('Jitter:DDP'), np.str_('MDVP:Shimmer'), np.str_('Shimmer:APQ3'), np.str_('Shimmer:APQ5'), np.str_('NHR'), np.str_('RPDE'), np.str_('spread1'), np.str_('PPE')]
  mean=0.9484 std=0.0540

Gaussian NB
  Features: [np.str_('MDVP:Fo(Hz)'), np.str_('MDVP:Fhi(Hz)'), np.str_('MDVP:Flo(Hz)'), np.str_('Shimmer:APQ3'), n

/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/python/3.12.1/lib